In [ ]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/creditFraudData/creditcard.csv")
print(df.shape)
df.head()

In [ ]:
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True) * 100)
print(df.isnull().sum().sum())

In [ ]:
df.groupby("Class")["Amount"].mean()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(np.log10(df["Amount"] + 1), bins=100, kde=True, color="purple", ax=axes[0])
axes[0].set_title("Distribution of Amount (Log Scale)")
axes[0].set_xlabel("Log(Amount + 1)")

sns.histplot(df["Time"], bins=100, color="teal", ax=axes[1])
axes[1].set_title("Distribution of Time (seconds elapsed)")
axes[1].set_xlabel("Time")

plt.tight_layout()
plt.show()

In [ ]:
# 1. Does fraud cluster in the overnight troughs?
sns.histplot(data=df, x="Time", hue="Class", bins=100, stat="density", common_norm=False)
plt.title("Time Distribution: Fraud vs Legitimate (normalized)")
plt.show()

# 2. Does that $1 spike correlate with fraud?
df[df["Amount"] < 5]["Class"].value_counts()

In [ ]:
df["Hour"] = (df["Time"] % 86400) // 3600

fraud_rate_by_hour = df.groupby("Hour")["Class"].mean()
print(fraud_rate_by_hour)

fraud_rate_by_hour.plot(kind="bar", figsize=(10,4), color="salmon")
plt.title("Fraud Rate by Hour of Day")
plt.ylabel("Fraud Rate")
plt.xlabel("Hour (0-23)")
plt.show()

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

df["IsNight"] = df["Hour"].between(1, 5)

night_fraud = df[df["IsNight"]]["Class"].sum()
night_total = df[df["IsNight"]].shape[0]

day_fraud = df[~df["IsNight"]]["Class"].sum()
day_total = df[~df["IsNight"]].shape[0]

print(f"Night: {night_fraud} fraud out of {night_total} ({night_fraud/night_total*100:.3f}%)")
print(f"Day:   {day_fraud} fraud out of {day_total} ({day_fraud/day_total*100:.3f}%)")

stat, pval = proportions_ztest([night_fraud, day_fraud], [night_total, day_total])
print(f"z-statistic: {stat:.3f}, p-value: {pval:.10f}")

In [ ]:
df_model = df.drop(["Time"], axis=1)
df_model.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_model.drop("Class", axis=1)
y = df_model["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", max_depth=10, random_state=42, n_jobs=-1))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = cross_val_score(rf_pipeline, X, y, cv=cv, scoring="f1")

print("F1 per fold:", f1_scores)
print("Mean F1:", f1_scores.mean())
print("Std deviation:", f1_scores.std())

In [ ]:
importances = rf.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print(importance_df.head(10))

plt.figure(figsize=(8,6))
sns.barplot(data=importance_df.head(10), x="importance", y="feature")
plt.title("Random Forest Feature Importance (Top 10)")
plt.show()

In [ ]:
def rule_low_amount(row):
    return row["Amount"] < 5

def rule_night_hour(row):
    return 1 <= row["Hour"] <= 5

df_model["rule_low_amount"] = df_model.apply(rule_low_amount, axis=1)
df_model["rule_night_hour"] = df_model.apply(rule_night_hour, axis=1)

df_model[["Amount", "Hour", "rule_low_amount", "rule_night_hour", "Class"]].head(10)

In [ ]:
print("Rule 1 (low amount) - fraud rate when TRUE:", df_model[df_model["rule_low_amount"]]["Class"].mean())
print("Rule 2 (night hour) - fraud rate when TRUE:", df_model[df_model["rule_night_hour"]]["Class"].mean())
print("Both rules TRUE - fraud rate:", df_model[df_model["rule_low_amount"] & df_model["rule_night_hour"]]["Class"].mean())
print("Both rules TRUE - count:", (df_model["rule_low_amount"] & df_model["rule_night_hour"]).sum())

In [ ]:
def compute_risk_score(ml_probability, low_amount, night_hour):
    boost = 0.0
    if low_amount and night_hour:
        boost = 0.20   # compounding effect confirmed: 12.8x baseline fraud rate when both trigger
    elif night_hour:
        boost = 0.10   # confirmed: ~4.2x baseline alone
    elif low_amount:
        boost = 0.05   # confirmed: ~1.9x baseline alone

    return min(ml_probability + boost, 1.0)


In [ ]:
y_probs_rf = rf.predict_proba(X_test_scaled)[:, 1]

X_test_with_rules = X_test.copy()
X_test_with_rules["ml_probability"] = y_probs_rf
X_test_with_rules["rule_low_amount"] = df_model.loc[X_test.index, "rule_low_amount"]
X_test_with_rules["rule_night_hour"] = df_model.loc[X_test.index, "rule_night_hour"]

X_test_with_rules["final_risk_score"] = X_test_with_rules.apply(
    lambda row: compute_risk_score(row["ml_probability"], row["rule_low_amount"], row["rule_night_hour"]),
    axis=1
)

X_test_with_rules[["Amount", "Hour", "ml_probability", "rule_low_amount", "rule_night_hour", "final_risk_score"]].head(10)

In [ ]:
from sklearn.metrics import classification_report

y_test_aligned = y_test.loc[X_test_with_rules.index]

# Using raw ML probability at 0.5 cutoff (this is just rf.predict(), same as before)
pred_ml_only = (X_test_with_rules["ml_probability"] >= 0.5).astype(int)

# Using the rule-boosted final risk score at 0.5 cutoff
pred_with_rules = (X_test_with_rules["final_risk_score"] >= 0.5).astype(int)

print("=== ML probability alone ===")
print(classification_report(y_test_aligned, pred_ml_only))

print("=== ML probability + rule boost ===")
print(classification_report(y_test_aligned, pred_with_rules))

In [ ]:
def compute_risk_score(ml_probability, low_amount, night_hour):
    boost = 0.0
    if low_amount and night_hour:
        boost = 0.08
    elif night_hour:
        boost = 0.05
    elif low_amount:
        boost = 0.02

    return min(ml_probability + boost, 1.0)

X_test_with_rules["final_risk_score"] = X_test_with_rules.apply(
    lambda row: compute_risk_score(row["ml_probability"], row["rule_low_amount"], row["rule_night_hour"]),
    axis=1
)

pred_with_rules_tuned = (X_test_with_rules["final_risk_score"] >= 0.5).astype(int)

print("=== ML probability + tuned rule boost ===")
print(classification_report(y_test_aligned, pred_with_rules_tuned))

In [ ]:
def assign_tier(ml_probability, low_amount, night_hour):
    if ml_probability >= 0.7:
        tier = "high"
    elif ml_probability >= 0.3:
        tier = "medium"
    else:
        tier = "low"

    # Escalation: a low-confidence transaction that ALSO trips both rules
    # gets bumped up one tier, since we proved that combination carries real signal
    if tier == "low" and low_amount and night_hour:
        tier = "medium"

    return tier

X_test_with_rules["tier"] = X_test_with_rules.apply(
    lambda row: assign_tier(row["ml_probability"], row["rule_low_amount"], row["rule_night_hour"]),
    axis=1
)

print(X_test_with_rules["tier"].value_counts())
print(X_test_with_rules["tier"].value_counts(normalize=True) * 100)

In [ ]:
tier_vs_fraud = pd.crosstab(X_test_with_rules["tier"], y_test_aligned, normalize="index") * 100
print(tier_vs_fraud)

print()
print("Actual fraud count per tier:")
print(pd.crosstab(X_test_with_rules["tier"], y_test_aligned))

In [ ]:
import os
from groq import Groq
import google.generativeai as genai
import time

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
gemini_model = genai.GenerativeModel("gemini-1.5-flash")


def call_groq(prompt):
    start = time.time()
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        max_tokens=600,
        messages=[{"role": "user", "content": prompt}]
    )
    latency_ms = (time.time() - start) * 1000
    text = response.choices[0].message.content

    if not text:
        text = "Explanation unavailable (model reasoning exceeded token budget)."

    return {
        "text": text,
        "latency_ms": round(latency_ms, 1),
        "cost_usd": 0.0,
        "tokens_used": response.usage.total_tokens,
        "provider": "groq/gpt-oss-20b",
        "tier": "medium"
    }


def call_gemini(prompt):
    start = time.time()
    response = gemini_model.generate_content(prompt)  # gemini_model built with "gemini-3.6-flash"
    latency_ms = (time.time() - start) * 1000
    usage = response.usage_metadata
    return {
        "text": response.text,
        "latency_ms": round(latency_ms, 1),
        "cost_usd": 0.0,
        "tokens_used": usage.total_token_count,
        "provider": "gemini-3.6-flash",
        "tier": "high"
    }


def build_prompt(row):
    return (
        f"A transaction was flagged as risk (model score: {row['ml_probability']:.2f}). "
        f"Amount: ${row['Amount']:.2f}, Hour: {int(row['Hour'])}:00. "
        f"Low-amount flag: {row['rule_low_amount']}. Night-hour flag: {row['rule_night_hour']}. "
        "In 2-3 plain-English sentences, explain the risk factors a fraud analyst should know, "
        "without claiming to know the model's internal reasoning."
    )


def get_explanation(row):
    if row["tier"] == "high":
        return call_gemini(build_prompt(row))
    elif row["tier"] == "medium":
        return call_groq(build_prompt(row))
    else:
        return {"text": None, "latency_ms": 0.0, "cost_usd": 0.0, "tokens_used": 0, "provider": None, "tier": "low"}

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads your .env file into environment variables

print("GROQ_API_KEY loaded:", os.getenv("GROQ_API_KEY") is not None)
print("GEMINI_API_KEY loaded:", os.getenv("GEMINI_API_KEY") is not None)

In [ ]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

In [ ]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Hello!"
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
from groq import Groq

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

test_response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    max_tokens=100,
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(test_response.choices[0].message.content)
print("Tokens used:", test_response.usage.total_tokens)

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
gemini_model = genai.GenerativeModel("gemini-3.6-flash")

test_response = gemini_model.generate_content("Say hello in one sentence.")
print(test_response.text)
print("Tokens used:", test_response.usage_metadata.total_token_count)

In [ ]:
sample_high = X_test_with_rules[X_test_with_rules["tier"] == "high"].iloc[0]
result = get_explanation(sample_high)
print(result)

sample_medium = X_test_with_rules[X_test_with_rules["tier"] == "medium"].iloc[0]
result2 = get_explanation(sample_medium)
print(result2)

In [ ]:
sample = pd.concat([
    X_test_with_rules[X_test_with_rules["tier"] == "high"].head(3),
    X_test_with_rules[X_test_with_rules["tier"] == "medium"].head(3),
    X_test_with_rules[X_test_with_rules["tier"] == "low"].head(3),
])

results = []
for idx, row in sample.iterrows():
    explanation = get_explanation(row)
    results.append({
        "index": idx,
        "tier": row["tier"],
        "amount": row["Amount"],
        "hour": row["Hour"],
        "ml_probability": row["ml_probability"],
        "actual_class": y_test_aligned.loc[idx],
        **explanation
    })

results_df = pd.DataFrame(results)
pd.set_option("display.max_colwidth", 100)
print(results_df[["tier", "amount", "hour", "ml_probability", "actual_class", "text", "latency_ms", "cost_usd", "provider"]])

In [ ]:
raw_response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    max_tokens=300,
    messages=[{"role": "user", "content": build_prompt(sample.iloc[3])}]
)
print(raw_response)

In [ ]:
for i in range(3):
    start = time.time()
    r = gemini_model.generate_content("Say hello in one sentence.")
    print(f"Call {i+1}: {(time.time()-start)*1000:.0f} ms")

In [ ]:
from sklearn.pipeline import Pipeline

final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", max_depth=10, random_state=42, n_jobs=-1))
])

final_pipeline.fit(X_train, y_train)  # note: unscaled X_train — the pipeline scales internally now

import joblib
import os
os.makedirs("../models", exist_ok=True)
joblib.dump(final_pipeline, "../models/fraud_model.joblib")